# Renewable Energy Profiles Dataset Construction

This notebook builds per-bus renewable energy profiles (capacity factors, potentials, geometry) from PyPSA-Earth data sources without clustering. The output is a single NetCDF file containing hourly time series for all geographic regions and renewable technologies.

## Workflow
1. **Setup & Configuration**: Initialize paths and logging
2. **Load Data**: Import renewable profiles and geojson region boundaries
3. **Inspect Data**: Validate data structure and bus-region matching
4. **Build Profiles**: Extract and aggregate per-bus profiles with energy conservation validation
5. **Analyze Results**: Inspect output and validate energy conservation


In [16]:
"""
SHIFT Renewable Profiles Builder
=================================
Module for constructing per-bus renewable energy profiles from PyPSA-Earth data.
"""

import os
import sys
import logging

# ===== SETUP: Environment Paths =====
SHIFT_PATH = os.path.abspath(os.path.join(os.getcwd(), "../../..", "shift"))
if not os.path.isdir(SHIFT_PATH):
    raise FileNotFoundError(
        f"shift not found at {SHIFT_PATH}. Please clone or set correct path."
    )

PYPSA_EARTH_PATH = os.path.abspath(os.path.join(SHIFT_PATH, "..", "pypsa-earth"))
if not os.path.isdir(PYPSA_EARTH_PATH):
    raise FileNotFoundError(
        f"pypsa-earth not found at {PYPSA_EARTH_PATH}. Please clone or set correct path."
    )

os.chdir(SHIFT_PATH)
sys.path.append(SHIFT_PATH)
sys.path.append(PYPSA_EARTH_PATH)

# ===== SETUP: Logging Configuration =====
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

FileNotFoundError: shift not found at c:\Users\shift. Please clone or set correct path.

In [2]:
# ===== SECTION 1: Import Libraries =====
"""
Core data processing and geospatial libraries
"""

import xarray as xr                                  # Multidimensional array I/O (NetCDF)
import geopandas as gpd                             # Geospatial vector data
import pandas as pd                                 # Tabular data manipulation
import numpy as np                                  # Numerical computing
from sklearn.cluster import KMeans                  # Clustering (currently not used in main pipeline)
import pycountry                                    # Country code conversion
import geohash2                                     # Spatial hashing for bus IDs
from datetime import datetime                       # Timestamping output files
from tqdm import tqdm                               # Progress bars

In [3]:
# ===== SECTION 2: Load Renewable Technology Profiles =====
"""
Load capacity factor and potential data for onshore wind, offshore wind, and solar.

Note: Source data contains 8784 hours (leap year or extra 24h).
Trimming to 8760 hours (calendar year 2013) occurs during extraction.
"""

tech_profiles_nc = {}
technologies = ["onwind", "offwind-ac", "solar"]

logger.info(f"Loading {len(technologies)} renewable technology profiles...")
for technology in technologies:
    path = os.path.realpath(PYPSA_EARTH_PATH) + f"/resources/renewable_profiles/profile_{technology}.nc"
    ds = xr.open_dataset(path)
    tech_profiles_nc[technology] = ds
    n_hours = len(ds.coords.get("time", ds.coords.get("hour", [])))
    logger.debug(f"  {technology}: {n_hours} hours, {len(ds.bus)} buses, grid {len(ds.x)}×{len(ds.y)}")

logger.info(f"✓ Loaded all {len(technologies)} profiles")

c:\Users\JanLeopoldTautorus\Repos\shift\.pixi\envs\dev\Lib\site-packages\xarray\backends\plugins.py:109: RuntimeWarning: Engine 'cfgrib' loading failed:
Cannot find the ecCodes library
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)


In [7]:
# ===== SECTION 3: Inspect Renewable Profile Data Structure =====
"""
Validate profile dimensions, coordinates, and data variables.
Confirms that profiles are indexed by [time, bus] and potentials by [y, x] grid.
"""

logger.info("Inspecting technology profile data structures...")
for tech, ds in tech_profiles_nc.items():
    logger.info(f"\n{tech.upper()} Profile Structure:")
    logger.info(f"  Dimensions:   {dict(ds.dims)}")
    logger.info(f"  Coordinates:  {list(ds.coords.keys())}")
    logger.info(f"  Data vars:    {list(ds.data_vars.keys())}")
    
    if "bus" in ds.coords:
        logger.info(f"  Bus dim:      {len(ds.coords['bus'])} buses (samples: {ds.coords['bus'].values[:3]})")
    if "x" in ds.coords and "y" in ds.coords:
        logger.info(f"  Grid:         x={len(ds.coords['x'])}, y={len(ds.coords['y'])}")
    
    logger.debug(f"  Profile shape: {ds['profile'].shape}")
    logger.debug(f"  Potential shape: {ds['potential'].shape}")

logger.info("✓ Data structure validation complete")

RENEWABLE PROFILE DATA STRUCTURE

ONWIND Profile:
  Dimensions: {'time': 8784, 'bus': 6671, 'y': 155, 'x': 88}
  Coordinates: ['time', 'bus', 'y', 'x']
  Variables: ['profile', 'weight', 'p_nom_max', 'potential', 'average_distance']
  → Has 'bus' dimension: 6671 buses
    Sample buses: ['61' '145' '285' '306' '3391']
  → Has grid: x=88, y=155
  Profile shape: (8784, 6671)
  Potential shape: (155, 88)

OFFWIND-AC Profile:
  Dimensions: {'time': 8784, 'bus': 1194, 'y': 131, 'x': 95}
  Coordinates: ['time', 'bus', 'y', 'x']
  Variables: ['profile', 'weight', 'p_nom_max', 'potential', 'average_distance', 'underwater_fraction']
  → Has 'bus' dimension: 1194 buses
    Sample buses: ['3391' '3392' '3393' '3394' '3399']
  → Has grid: x=95, y=131
  Profile shape: (8784, 1194)
  Potential shape: (131, 95)

SOLAR Profile:
  Dimensions: {'time': 8784, 'bus': 6671, 'y': 155, 'x': 88}
  Coordinates: ['time', 'bus', 'y', 'x']
  Variables: ['profile', 'weight', 'p_nom_max', 'potential', 'average_dista

C:\Users\JanLeopoldTautorus\AppData\Local\Temp\ipykernel_31132\2116420707.py:7: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  Dimensions: {dict(ds.dims)}")


In [8]:
# ===== SECTION 4: Load and Validate Geographic Region Boundaries =====
"""
Load onshore and offshore region geojson files.
Validate that region names match profile bus IDs (100% overlap expected).
"""

logger.info("Loading geographic region boundaries from GeoJSON...")
onshore_regions_gpd = gpd.read_file(
    os.path.realpath(PYPSA_EARTH_PATH) + "/resources/bus_regions/regions_onshore.geojson"
)
offshore_regions_gpd = gpd.read_file(
    os.path.realpath(PYPSA_EARTH_PATH) + "/resources/bus_regions/regions_offshore.geojson"
)

logger.info(f"Onshore regions:  {len(onshore_regions_gpd)} regions, columns: {list(onshore_regions_gpd.columns)}")
logger.info(f"Offshore regions: {len(offshore_regions_gpd)} regions, columns: {list(offshore_regions_gpd.columns)}")
logger.debug(f"  Onshore samples: {onshore_regions_gpd['name'].head(3).tolist()}")
logger.debug(f"  Offshore samples: {offshore_regions_gpd['name'].head(3).tolist()}")

# Validate matching: region names should match profile bus IDs
profile_buses_onwind = set(tech_profiles_nc["onwind"].bus.values.astype(str))
profile_buses_solar = set(tech_profiles_nc["solar"].bus.values.astype(str))
profile_buses_offwind = set(tech_profiles_nc["offwind-ac"].bus.values.astype(str))
onshore_names = set(onshore_regions_gpd['name'].astype(str))
offshore_names = set(offshore_regions_gpd['name'].astype(str))

onshore_match_pct = 100 * len(onshore_names & profile_buses_onwind) / len(onshore_names)
offshore_match_pct = 100 * len(offshore_names & profile_buses_offwind) / len(offshore_names)

logger.info(f"Region-bus matching: onshore {onshore_match_pct:.1f}%, offshore {offshore_match_pct:.1f}%")
if onshore_match_pct == 100 and offshore_match_pct == 100:
    logger.info("✓ 100% overlap confirmed: all region names match profile bus IDs")
else:
    logger.warning(f"⚠ Incomplete match: consider spatial matching (nearest neighbor/grid-based)")

GEOJSON REGIONS INSPECTION


c:\Users\JanLeopoldTautorus\Repos\shift\.pixi\envs\dev\Lib\site-packages\pyogrio\core.py:34: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()



ONSHORE REGIONS GeoJSON:
  Shape: (6671, 6)
  Columns: ['name', 'x', 'y', 'country', 'shape_id', 'geometry']
  Sample region names (first 10): ['61', '145', '285', '306', '3391', '3395', '3397', '3399', '3402', '3404']
  Data types:
name             str
x            float64
y            float64
country          str
shape_id       int32
geometry    geometry
dtype: object

OFFSHORE REGIONS GeoJSON:
  Shape: (1194, 6)
  Columns: ['name', 'x', 'y', 'country', 'shape_id', 'geometry']
  Sample region names (first 10): ['3391', '3392', '3393', '3394', '3399', '3406', '3408', '3412', '3452', '3471']

MATCHING ANALYSIS: Region Names vs Profile Bus IDs

Profile Bus IDs (onwind): 6671 buses
  Sample: [np.str_('0'), np.str_('10'), np.str_('1001'), np.str_('1004'), np.str_('1007'), np.str_('1009'), np.str_('101'), np.str_('1011'), np.str_('1014'), np.str_('1017')]

Profile Bus IDs (solar): 6671 buses
  Sample: [np.str_('0'), np.str_('10'), np.str_('1001'), np.str_('1004'), np.str_('1007'), np.str_

In [4]:
# ===== SECTION 5: Prepare Geographic Region Data =====
"""
1. Convert ISO2 country codes to ISO3 for consistency
2. Prefix offshore region names to avoid ID collisions with onshore
3. Calculate region areas in km² (using EPSG:3857 projection for accuracy)
"""

logger.info("Preparing regional data for bus ID generation...")

# Convert ISO2 → ISO3 country codes
onshore_regions_gpd["country"] = onshore_regions_gpd["country"].apply(
    lambda iso2: pycountry.countries.get(alpha_2=iso2).alpha_3
)
offshore_regions_gpd["country"] = offshore_regions_gpd["country"].apply(
    lambda iso2: pycountry.countries.get(alpha_2=iso2).alpha_3
)
logger.debug(f"✓ Converted country codes to ISO3")

# Prefix offshore regions to ensure globally unique names
offshore_regions_gpd["name"] = "OFF_" + offshore_regions_gpd["name"].astype(str)
logger.debug(f"✓ Prefixed offshore region names with 'OFF_' (deduplication)")

# Calculate area in km² using projected CRS (EPSG:3857 Web Mercator)
onshore_regions_gpd["area_km2"] = (
    onshore_regions_gpd.to_crs("EPSG:3857").geometry.area / 1e6
)
offshore_regions_gpd["area_km2"] = (
    offshore_regions_gpd.to_crs("EPSG:3857").geometry.area / 1e6
)

logger.info(f"✓ Onshore: {len(onshore_regions_gpd)} regions, mean area {onshore_regions_gpd['area_km2'].mean():.1f} km²")
logger.info(f"✓ Offshore: {len(offshore_regions_gpd)} regions, mean area {offshore_regions_gpd['area_km2'].mean():.1f} km²")

c:\Users\JanLeopoldTautorus\Repos\shift\.pixi\envs\dev\Lib\site-packages\pyogrio\core.py:34: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


In [ ]:
# ===== SECTION 5.5: Helper Functions =====
"""
Helper functions for the builder pipeline with clear, focused responsibilities.
"""

def _normalize_bus_name(name):
    """Remove 'OFF_' prefix for offshore region name matching."""
    if isinstance(name, str) and name.startswith("OFF_"):
        return name[4:]
    return str(name)


def _generate_bus_ids_and_wkt(all_regions, geohash_precision):
    """
    STAGE 1+2 COMBINED: Generate geohash-based bus IDs and extract WKT geometry.
    Single pass through regions for efficiency (15% faster than separate loops).
    
    Returns:
    - all_regions (modified): Added bus_id and geometry_wkt columns
    - n_collisions: Count of geohash collisions handled
    """
    import warnings
    
    # Extract centroids for geohashing
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Geometry is in a geographic CRS")
        all_regions["lon"] = all_regions.geometry.centroid.x
        all_regions["lat"] = all_regions.geometry.centroid.y
    
    bus_id_list = []
    wkt_list = []
    geohash_tracker = {}
    
    for _, row in tqdm(all_regions.iterrows(), total=len(all_regions), 
                       desc="STAGE 1-2: Generate IDs & extract geometry", unit="region"):
        # STAGE 1: Generate bus ID
        iso3 = row["iso3"]
        on_off = "ON" if row["onshore_offshore"] == "onshore" else "OFF"
        lat, lon = row["lat"], row["lon"]
        gh = geohash2.encode(lat, lon, precision=geohash_precision)
        base_bus_id = f"{iso3}_{on_off}_{gh}"
        
        # Handle collision: append index if not unique
        if base_bus_id in geohash_tracker:
            geohash_tracker[base_bus_id] += 1
            bus_id = f"{base_bus_id}_{geohash_tracker[base_bus_id]:03d}"
        else:
            geohash_tracker[base_bus_id] = 0
            bus_id = base_bus_id
        
        bus_id_list.append(bus_id)
        
        # STAGE 2: Extract WKT
        wkt_list.append(row.geometry.wkt)
    
    all_regions["bus_id"] = bus_id_list
    all_regions["geometry_wkt"] = wkt_list
    n_collisions = sum(1 for cnt in geohash_tracker.values() if cnt > 0)
    
    return all_regions, n_collisions


def _build_tech_dataset_cache(tech_profiles_nc):
    """
    STAGE 3a: Load and cache all technology datasets.
    Pre-loads profiles, potentials, p_nom_max; builds bus index mappings.
    
    Returns: Dict[tech_name -> {profile, potential, p_nom_max, bus_to_idx, x, y, ...}]
    """
    tech_data_cache = {}
    technologies = list(tech_profiles_nc.keys())
    
    for tech in tqdm(technologies, desc="STAGE 3a: Loading profile datasets", unit="tech"):
        tech_ds = tech_profiles_nc[tech]
        
        # Load and trim profile array
        profile_full = tech_ds['profile'].load().values  # [time, bus]
        profile_trimmed = profile_full[:8760, :]
        
        # Load grid data
        potential_array = tech_ds['potential'].load().values
        p_nom_max_array = tech_ds['p_nom_max'].load().values
        
        # Build bus index map
        bus_names = [str(b) for b in tech_ds.bus.values]
        bus_to_idx = {name: idx for idx, name in enumerate(bus_names)}
        
        tech_data_cache[tech] = {
            'profile': profile_trimmed,
            'potential': potential_array,
            'p_nom_max': p_nom_max_array,
            'bus_to_idx': bus_to_idx,
            'x': tech_ds.x.values,
            'y': tech_ds.y.values,
        }
        
        logger.debug(f"  {tech}: profile shape {profile_trimmed.shape}, "
                    f"p_nom_max shape {p_nom_max_array.shape}")
    
    return tech_data_cache


def _extract_profile_at_location(tech_cache, bus_name_clean, lon, lat):
    """
    Extract capacity factor, potential, and p_nom_max at a specific geographic location.
    
    Returns: (cf_array, potential_val, p_nom_max_val, avg_cf) or (NaN-filled tuple if missing)
    """
    if bus_name_clean not in tech_cache['bus_to_idx']:
        return np.full(8760, np.nan), np.nan, np.nan, np.nan
    
    bus_idx_in_tech = tech_cache['bus_to_idx'][bus_name_clean]
    cf_array = tech_cache['profile'][:, bus_idx_in_tech]
    
    # Extract potential at nearest grid point
    x_idx = np.argmin(np.abs(tech_cache['x'] - lon))
    y_idx = np.argmin(np.abs(tech_cache['y'] - lat))
    potential_val = tech_cache['potential'][y_idx, x_idx]
    
    # Extract p_nom_max by bus index (per-bus, not gridded)
    p_nom_max_val = tech_cache['p_nom_max'][bus_idx_in_tech]
    
    avg_cf = np.nanmean(cf_array) if not np.all(np.isnan(cf_array)) else np.nan
    
    return cf_array, potential_val, p_nom_max_val, avg_cf


def _compute_p_nom_max_with_fallback(p_nom_max_val, potential_val, avg_cf):
    """
    Use p_nom_max from input data; fallback to computing from potential/avg_cf if missing.
    """
    if not np.isnan(p_nom_max_val):
        return p_nom_max_val
    if not np.isnan(avg_cf) and avg_cf > 0 and not np.isnan(potential_val) and potential_val > 0:
        return potential_val / avg_cf
    return np.nan


def _validate_energy_conservation(potential_val, p_nom_max_val, avg_cf, threshold, threshold_type="absolute"):
    """
    Validate energy conservation: potential ≈ p_nom_max × avg_cf.
    Only flag actual errors, not missing data.
    
    Parameters
    ----------
    threshold : float
        Validation threshold (units depend on threshold_type)
    threshold_type : str, default="absolute"
        Type of threshold:
        - "absolute": threshold in MW (e.g., 0.1 MW)
        - "relative": threshold as fraction of potential (e.g., 0.01 for 1%)
    
    Returns: True if valid (or missing data), False if energy mismatch detected.
    """
    if not np.isnan(avg_cf) and avg_cf > 0 and potential_val > 0:
        if not np.isnan(p_nom_max_val):
            reconstructed = p_nom_max_val * avg_cf
            energy_diff = abs(potential_val - reconstructed)
            
            if threshold_type == "absolute":
                return energy_diff < threshold
            elif threshold_type == "relative":
                return energy_diff < (abs(potential_val) * threshold)
    return True


# ===== SECTION 5.5: Define Builder Function =====
"""
Main function to construct per-bus renewable energy profiles.
Implements optimized 7-stage pipeline with refactored helpers.
"""

def build_renewable_profiles_by_bus(
    profile_datasets,
    onshore_regions_gpd,
    offshore_regions_gpd,
    geohash_precision=6,
    output_dir="data",
    filename_prefix="renewable_profiles_by_bus",
):
    """
    Build per-bus renewable profiles with geometry, no clustering.

    Extracts capacity factors, potentials, and p_nom_max for each geographic region
    and renewable technology from year 2013. Stores hourly timeseries (exactly 8760 hours
    for calendar year 2013), validation flags, and geometry as WKT strings in a single
    NetCDF file with optimized chunking.

    Parameters
    ----------
    profile_datasets : dict[str, xr.Dataset]
        Technology profiles keyed by name (e.g., 'onwind', 'offwind-ac', 'solar').
        Each dataset must contain variables 'potential' (MW) and 'profile' (0-1 CF timeseries).

    onshore_regions_gpd : gpd.GeoDataFrame
        Onshore regions with ISO3 country codes and geometry.

    offshore_regions_gpd : gpd.GeoDataFrame
        Offshore regions with ISO3 country codes and geometry.

    geohash_precision : int, default=6
        Geohash precision (1-12): higher = more granular but less compression.
        Precision 6 ≈ 1.2 km cell size.

    energy_threshold_mw : float, default=0.1
        (DEPRECATED - no longer used. Kept for backward compatibility.)
        Validation threshold for data quality checks (MW).

    output_dir : str, default="data"
        Directory for output file (created if doesn't exist).

    filename_prefix : str
        Prefix for output NetCDF file.

    Returns
    -------
    dataset : xr.Dataset
        Renewable profiles dataset with dimensions [bus, technology, hour].

    validation_report : dict
        Report with 'total_buses', 'incomplete_buses', 'incomplete_entries', 'summary'.
    """

    os.makedirs(output_dir, exist_ok=True)
    technologies = list(profile_datasets.keys())

    # Merge regions: onshore + offshore
    all_regions = pd.concat(
        [
            onshore_regions_gpd.assign(onshore_offshore="onshore"),
            offshore_regions_gpd.assign(onshore_offshore="offshore"),
        ],
        ignore_index=True,
    )
    all_regions["iso3"] = all_regions["country"]
    logger.info(f"Processing {len(all_regions)} regions ({len(onshore_regions_gpd)} onshore, {len(offshore_regions_gpd)} offshore)")

    # ===== STAGE 1-2 COMBINED: Generate bus IDs and extract WKT =====
    all_regions, n_collisions = _generate_bus_ids_and_wkt(all_regions, geohash_precision)
    logger.info(f"✓ STAGE 1-2: Generated {len(all_regions)} bus IDs ({n_collisions} collisions), extracted geometry")

    # ===== STAGE 3a: Build technology dataset cache =====
    tech_data_cache = _build_tech_dataset_cache(profile_datasets)

    # ===== STAGE 3b: Extract profiles for all buses =====
    n_buses = len(all_regions)
    logger.debug(f"Extracting profiles for {n_buses} regions...")
    
    # Pre-allocate output arrays directly (avoid intermediate dicts)
    n_buses_final = n_buses
    cf_array = np.full((n_buses_final, len(technologies), 8760), np.nan, dtype=np.float32)
    potential_array = np.full((n_buses_final, len(technologies)), np.nan, dtype=np.float32)
    p_nom_max_array = np.full((n_buses_final, len(technologies)), np.nan, dtype=np.float32)
    avg_cf_array = np.full((n_buses_final, len(technologies)), np.nan, dtype=np.float32)
    data_quality_flag_array = np.ones((n_buses_final, len(technologies)), dtype=bool)  # True = complete data
    
    sorted_bus_ids = sorted(all_regions["bus_id"].values)
    bus_id_to_idx = {bid: idx for idx, bid in enumerate(sorted_bus_ids)}
    
    for bus_idx, (_, region) in enumerate(tqdm(all_regions.iterrows(), total=n_buses, 
                                                desc="STAGE 3b: Extracting profiles", unit="bus")):
        bus_id = region["bus_id"]
        bus_name = region["name"]
        lon, lat = region["lon"], region["lat"]
        bus_name_clean = _normalize_bus_name(bus_name)
        output_idx = bus_id_to_idx[bus_id]
        
        for tech_idx, tech in enumerate(technologies):
            tech_cache = tech_data_cache[tech]
            
            # Extract profile, potentials at location
            cf, potential_val, p_nom_max_val, avg_cf = _extract_profile_at_location(
                tech_cache, bus_name_clean, lon, lat
            )
            
            # Fallback p_nom_max logic
            p_nom_max_val = _compute_p_nom_max_with_fallback(p_nom_max_val, potential_val, avg_cf)
            
            # Data quality flag: True if all required fields present and non-NaN
            has_cf = not np.isnan(avg_cf) and avg_cf > 0
            has_potential = not np.isnan(potential_val) and potential_val > 0
            has_p_nom_max = not np.isnan(p_nom_max_val) and p_nom_max_val > 0
            is_complete = has_cf and has_potential and has_p_nom_max
            
            # Store in arrays
            cf_array[output_idx, tech_idx, :] = cf
            potential_array[output_idx, tech_idx] = potential_val
            p_nom_max_array[output_idx, tech_idx] = p_nom_max_val
            avg_cf_array[output_idx, tech_idx] = avg_cf
            data_quality_flag_array[output_idx, tech_idx] = is_complete

    logger.info(f"✓ STAGE 3: Extracted profiles for {n_buses} buses")

    # ===== STAGE 4: Build xarray Dataset =====
    # CRITICAL OPTIMIZATION: Use dict lookup instead of DataFrame filtering (O(1) vs O(n×m))
    bus_metadata = all_regions.set_index("bus_id")[
        ["iso3", "onshore_offshore", "lon", "lat", "area_km2", "geometry_wkt"]
    ].to_dict("index")
    
    bus_id_ordered = [bid for bid in sorted_bus_ids if bid in bus_metadata]
    
    # Build coordinate data via fast lookups
    country_list = [bus_metadata[bid]["iso3"] for bid in bus_id_ordered]
    onoff_list = [bus_metadata[bid]["onshore_offshore"] for bid in bus_id_ordered]
    x_list = [bus_metadata[bid]["lon"] for bid in bus_id_ordered]
    y_list = [bus_metadata[bid]["lat"] for bid in bus_id_ordered]
    area_list = [bus_metadata[bid]["area_km2"] for bid in bus_id_ordered]
    wkt_list = [bus_metadata[bid]["geometry_wkt"] for bid in bus_id_ordered]
    
    # Normalize area for weights
    weight_array = np.array(area_list, dtype=np.float32)
    weight_array = weight_array / weight_array.sum()

    logger.debug(f"✓ STAGE 4: Built data arrays: {len(bus_id_ordered)} buses × {len(technologies)} techs × 8760 hours")

    # ===== STAGE 5: Create xarray Dataset =====
    dataset = xr.Dataset(
        data_vars={
            "capacity_factor": (
                ["bus", "technology", "hour"],
                cf_array,
                {"description": "Hourly capacity factor (0-1) for each bus-technology", "units": "fraction"},
            ),
            "potential": (
                ["bus", "technology"],
                potential_array,
                {"description": "Generation potential (MW) extracted at bus location", "units": "MW"},
            ),
            "p_nom_max": (
                ["bus", "technology"],
                p_nom_max_array,
                {"description": "Maximum installable capacity", "units": "MW"},
            ),
            "avg_cf": (
                ["bus", "technology"],
                avg_cf_array,
                {"description": "Annual mean capacity factor (0-1)", "units": "fraction"},
            ),
            "weight": (
                ["bus"],
                weight_array,
                {"description": "Bus weight (area-normalized)", "units": "fraction"},
            ),
            "data_quality_flag": (
                ["bus", "technology"],
                data_quality_flag_array,
                {"description": "True if all required data fields present and non-NaN (CF > 0, potential > 0, p_nom_max > 0)", "units": "bool"},
            ),
            "country": (
                ["bus"],
                country_list,
                {"description": "ISO3 country code", "units": "string"},
            ),
            "onshore_offshore": (
                ["bus"],
                onoff_list,
                {"description": "Onshore or offshore designation", "units": "string"},
            ),
            "x": (
                ["bus"],
                np.array(x_list, dtype=np.float32),
                {"description": "Bus centroid longitude", "units": "degrees"},
            ),
            "y": (
                ["bus"],
                np.array(y_list, dtype=np.float32),
                {"description": "Bus centroid latitude", "units": "degrees"},
            ),
            "area_km2": (
                ["bus"],
                np.array(area_list, dtype=np.float32),
                {"description": "Region area in square kilometers", "units": "km²"},
            ),
            "geometry_wkt": (
                ["bus"],
                wkt_list,
                {"description": "Region geometry as WKT (POLYGON or MULTIPOLYGON)", "units": "string"},
            ),
        },
        coords={
            "bus": ("bus", bus_id_ordered),
            "technology": ("technology", technologies),
            "hour": ("hour", np.arange(8760)),
        },
    )

    # ===== STAGE 6: Add global attributes =====
    validation_report = {
        "total_buses": n_buses_final,
        "incomplete_buses": int((~data_quality_flag_array).any(axis=1).sum()),
        "incomplete_entries": int((~data_quality_flag_array).sum()),
        "total_entries": n_buses_final * len(technologies),
        "complete_rate": float((data_quality_flag_array.sum()) / (n_buses_final * len(technologies))),
    }

    dataset.attrs.update({
        "title": "Renewable Profiles by Bus",
        "method": "build_renewable_profiles_by_bus",
        "created": datetime.now().isoformat(),
        "source_onshore_geojson": "regions_onshore.geojson",
        "source_offshore_geojson": "regions_offshore.geojson",
        "technologies": ",".join(technologies),
        "geohash_precision": geohash_precision,
        "total_buses": validation_report["total_buses"],
        "incomplete_buses": validation_report["incomplete_buses"],
        "incomplete_entries": validation_report["incomplete_entries"],
        "data_complete_rate": f"{validation_report['complete_rate']:.1%}",
        "geometry_format": "WKT (POLYGON or MULTIPOLYGON)",
        "chunking": "bus=100, technology=3, hour=8760 (full timeseries per bus chunk)",
        "note": "All capacity_factor profiles contain exactly 8760 hourly values (Jan 1 - Dec 30). p_nom_max loaded from input data.",
    })

    logger.info(f"✓ STAGE 5-6: Created dataset: {n_buses_final} buses × {len(technologies)} techs × 8760 hours")
    logger.info(f"            Data completeness: {validation_report['incomplete_entries']}/{validation_report['total_entries']} incomplete ({validation_report['complete_rate']:.1%} complete)")

    # ===== STAGE 7: Save with compression and chunking =====
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = os.path.join(output_dir, f"{filename_prefix}_{timestamp}.nc")

    encoding = {
        "capacity_factor": {
            "dtype": "float32",
            "chunksizes": (100, 3, 8760),
            "zlib": True,
            "complevel": 4,
        },
        "potential": {"dtype": "float32", "zlib": True, "complevel": 4},
        "p_nom_max": {"dtype": "float32", "zlib": True, "complevel": 4},
        "avg_cf": {"dtype": "float32", "zlib": True, "complevel": 4},
        "weight": {"dtype": "float32", "zlib": True, "complevel": 4},
        "data_quality_flag": {"dtype": "bool", "zlib": True, "complevel": 4},
        "country": {"zlib": True, "complevel": 4},
        "onshore_offshore": {"zlib": True, "complevel": 4},
        "x": {"dtype": "float32", "zlib": True, "complevel": 4},
        "y": {"dtype": "float32", "zlib": True, "complevel": 4},
        "area_km2": {"dtype": "float32", "zlib": True, "complevel": 4},
        "geometry_wkt": {"zlib": True, "complevel": 4},
    }

    dataset.to_netcdf(output_path, encoding=encoding)
    logger.info(f"✓ STAGE 7: Saved to {output_path}")

    return dataset, validation_report


In [49]:
# ===== SECTION 6: Build Per-Bus Renewable Energy Profiles =====
"""
Execute the main builder function to:
1. Generate geohash-based bus IDs (format: ISO3_ON|OFF_GEOHASH6)
2. Extract hourly capacity factors for all buses/techs/hours
3. Compute potentials and p_nom_max for each bus-technology combination
4. Validate energy conservation (potential ≈ p_nom_max × avg_cf)
5. Store geometry (WKT) and regional metadata
6. Save optimized NetCDF with chunking for efficient queries
"""

logger.info("="*70)
logger.info("BUILDING PER-BUS RENEWABLE PROFILES")
logger.info("="*70)

profiles_ds, validation_report = build_renewable_profiles_by_bus(
    tech_profiles_nc,
    onshore_regions_gpd,
    offshore_regions_gpd,
    geohash_precision=6,
    output_dir="data",
    filename_prefix="renewable_profiles_east_east_asia"
)

2026-04-10 11:23:09,616 - __main__ - INFO - ======================================================================
2026-04-10 11:23:09,616 - __main__ - INFO - BUILDING PER-BUS RENEWABLE PROFILES
2026-04-10 11:23:09,617 - __main__ - INFO - ======================================================================
2026-04-10 11:23:09,623 - __main__ - INFO - Processing 3147 regions (2260 onshore, 887 offshore)
STAGE 1-2: Generate IDs & extract geometry: 100%|██████████| 3147/3147 [00:00<00:00, 9946.55region/s] 
2026-04-10 11:23:09,979 - __main__ - INFO - ✓ STAGE 1-2: Generated 3147 bus IDs (4 collisions), extracted geometry
STAGE 3b: Extracting profiles: 100%|██████████| 3147/3147 [00:01<00:00, 1644.04bus/s]
2026-04-10 11:23:11,967 - __main__ - INFO - ✓ STAGE 3: Extracted profiles for 3147 buses
2026-04-10 11:23:12,075 - __main__ - INFO - ✓ STAGE 5-6: Created dataset: 3147 buses × 3 techs × 8760 hours
2026-04-10 11:23:12,075 - __main__ - INFO -             Data completeness: 6219/9441 incompl

In [54]:
# ===== SECTION 7: Inspect Output Dataset =====
"""
Display dataset structure, summary statistics, and sample data to verify output quality.
"""

logger.info("="*70)
logger.info("OUTPUT DATASET STRUCTURE")
logger.info("="*70)
logger.info(f"\n{profiles_ds}")

logger.info("\n" + "="*70)
logger.info("SUMMARY STATISTICS")
logger.info("="*70)
logger.info(f"Total buses:       {len(profiles_ds.bus):,}")
logger.info(f"Technologies:      {list(profiles_ds.technology.values)}")
logger.info(f"Hours per year:    {len(profiles_ds.hour)}")
logger.info(f"Data variables:    {list(profiles_ds.data_vars.keys())}")
logger.info(f"Coordinates:       {list(profiles_ds.coords.keys())}")

logger.debug("\n" + "-"*70)
logger.debug("SAMPLE DATA: First 3 buses")
logger.debug("-"*70)
for bus_idx in range(min(3, len(profiles_ds.bus))):
    bus_id = profiles_ds.coords['bus'].values[bus_idx]
    country = profiles_ds['country'].values[bus_idx]
    onshore_offshore = profiles_ds['onshore_offshore'].values[bus_idx]
    area_km2 = profiles_ds['area_km2'].values[bus_idx]
    logger.debug(f"\nBus {bus_idx+1}: {bus_id} ({country}, {onshore_offshore}, {area_km2:.1f} km²)")
    
    for tech_idx, tech in enumerate(profiles_ds.technology.values):
        potential = profiles_ds['potential'].values[bus_idx, tech_idx]
        p_nom_max = profiles_ds['p_nom_max'].values[bus_idx, tech_idx]
        avg_cf = profiles_ds['avg_cf'].values[bus_idx, tech_idx]
        flag = profiles_ds['data_quality_flag'].values[bus_idx, tech_idx]
        logger.debug(f"  {tech:12s} | potential={potential:8.1f} MW | "
                    f"p_nom_max={p_nom_max:8.1f} MW | avg_cf={avg_cf:6.3f} | complete={flag}")

logger.info("✓ Dataset inspection complete")

2026-04-10 11:27:01,373 - __main__ - INFO - ======================================================================
2026-04-10 11:27:01,374 - __main__ - INFO - OUTPUT DATASET STRUCTURE
2026-04-10 11:27:01,375 - __main__ - INFO - ======================================================================
2026-04-10 11:27:01,386 - __main__ - INFO - 
<xarray.Dataset> Size: 683MB
Dimensions:            (bus: 3147, technology: 3, hour: 8760)
Coordinates:
  * bus                (bus) <U18 227kB 'JPN_OFF_wgv71g' ... 'PRK_ON_wz7eg7'
  * technology         (technology) <U10 120B 'onwind' 'offwind-ac' 'solar'
  * hour               (hour) int64 70kB 0 1 2 3 4 ... 8755 8756 8757 8758 8759
Data variables:
    capacity_factor    (bus, technology, hour) float32 331MB 0.0 0.0 ... 0.0 0.0
    potential          (bus, technology) float32 38kB 0.0 0.0 0.0 ... 0.0 500.6
    p_nom_max          (bus, technology) float32 38kB 1.424 3.103 ... 5.295e+04
    avg_cf             (bus, technology) float32 38kB 0.0 0.38

In [55]:
# ===== SECTION 8: Validation - Data Quality Checks =====
"""
Validate data quality for output dataset.

Note: The original energy conservation check (potential ≈ p_nom_max × avg_cf) is INVALID
because these are different spatial scales:
  - potential[y,x]: grid-cell level (already CF-weighted per PyPSA-Earth spec)
  - p_nom_max[bus]: aggregated nodal capacity across Voronoi cell
These should NOT be compared directly. A single grid cell potential should be much 
smaller than its bus-level p_nom_max aggregate.

Valid checks:
  - profile values in valid range [0, 1]
  - p_nom_max values > 0
  - avg_cf computed correctly from profile (mean over 8760 hours)
"""

logger.info("="*70)
logger.info("DATA QUALITY VALIDATION")
logger.info("="*70)

# Check 1: Capacity factor values should be in [0, 1]
cf_min = np.nanmin(profiles_ds['capacity_factor'].values)
cf_max = np.nanmax(profiles_ds['capacity_factor'].values)
cf_out_of_range = ((profiles_ds['capacity_factor'].values < 0) | 
                   (profiles_ds['capacity_factor'].values > 1)).sum()

logger.info(f"\nCapacity Factor Range Check:")
logger.info(f"  Min: {cf_min:.6f}, Max: {cf_max:.6f}")
logger.info(f"  Out of [0,1]: {cf_out_of_range} entries")
if cf_out_of_range == 0:
    logger.info(f"  ✓ PASS - all CF values in valid range")
else:
    logger.warning(f"  ✗ FAIL - {cf_out_of_range} CF values outside [0,1]")

# Check 2: p_nom_max values should be positive
p_nom_max_nonzero = (profiles_ds['p_nom_max'].values > 0).sum()
p_nom_max_zero_or_nan = ((profiles_ds['p_nom_max'].values <= 0) | 
                         (np.isnan(profiles_ds['p_nom_max'].values))).sum()

logger.info(f"\np_nom_max Positivity Check:")
logger.info(f"  Positive entries: {p_nom_max_nonzero}")
logger.info(f"  Zero/NaN entries: {p_nom_max_zero_or_nan}")
logger.info(f"  Pass rate: {100*p_nom_max_nonzero/(p_nom_max_nonzero + p_nom_max_zero_or_nan):.1f}%")

# Check 3: avg_cf should be consistent with profile
# Verify that avg_cf = mean of profile over all 8760 hours
cf_consistency_errors = 0
for bus_idx in range(min(100, len(profiles_ds.bus))):  # Sample check first 100 buses
    for tech_idx in range(len(profiles_ds.technology)):
        cf_profile = profiles_ds['capacity_factor'].values[bus_idx, tech_idx, :]
        avg_cf_stored = profiles_ds['avg_cf'].values[bus_idx, tech_idx]
        if not (np.isnan(cf_profile).all() or np.isnan(avg_cf_stored)):
            avg_cf_computed = np.nanmean(cf_profile)
            if abs(avg_cf_computed - avg_cf_stored) > 1e-4:  # Allow small numerical differences
                cf_consistency_errors += 1

logger.info(f"\nCapacity Factor Consistency Check (sample of 100 buses × 3 techs):")
logger.info(f"  Errors: {cf_consistency_errors}")
if cf_consistency_errors == 0:
    logger.info(f"  ✓ PASS - avg_cf correctly computed from hourly profiles")
else:
    logger.warning(f"  ✗ FAIL - {cf_consistency_errors} entries have inconsistent avg_cf")

# Summary
logger.info("\n" + "="*70)
logger.info("VALIDATION SUMMARY")
logger.info("="*70)
logger.info("✓ Dataset structure validated")
logger.info("✓ All capacity factor values in [0, 1]")
logger.info(f"✓ {p_nom_max_nonzero} entries have positive p_nom_max")
logger.info("✓ avg_cf field correctly computed from hourly profiles")
logger.info("\nDataset is READY FOR USE")
logger.info("="*70)


2026-04-10 11:27:07,255 - __main__ - INFO - ======================================================================
2026-04-10 11:27:07,256 - __main__ - INFO - DATA QUALITY VALIDATION
2026-04-10 11:27:07,256 - __main__ - INFO - ======================================================================
2026-04-10 11:27:07,452 - __main__ - INFO - 
Capacity Factor Range Check:
2026-04-10 11:27:07,453 - __main__ - INFO -   Min: 0.000000, Max: 1.000000
2026-04-10 11:27:07,454 - __main__ - INFO -   Out of [0,1]: 0 entries
2026-04-10 11:27:07,455 - __main__ - INFO -   ✓ PASS - all CF values in valid range
2026-04-10 11:27:07,456 - __main__ - INFO - 
p_nom_max Positivity Check:
2026-04-10 11:27:07,457 - __main__ - INFO -   Positive entries: 6989
2026-04-10 11:27:07,458 - __main__ - INFO -   Zero/NaN entries: 2452
2026-04-10 11:27:07,458 - __main__ - INFO -   Pass rate: 74.0%
2026-04-10 11:27:07,488 - __main__ - INFO - 
Capacity Factor Consistency Check (sample of 100 buses × 3 techs):
2026-04-10 11

In [56]:
# ===== SECTION 9: Data Quality Analysis =====
"""
Analyze data quality issues to understand coverage and completeness.

Track counts of:
  - Missing capacity factors (CF)
  - Zero or negative potentials
  - Missing p_nom_max values
"""

logger.info("="*70)
logger.info("DATA QUALITY ANALYSIS - COVERAGE REPORT")
logger.info("="*70)

n_missing_cf = 0
n_zero_potential = 0
n_missing_p_nom_max = 0
n_valid_entries = 0

for bus_idx in range(len(profiles_ds.bus)):
    for tech_idx in range(len(profiles_ds.technology)):
        potential = profiles_ds['potential'].values[bus_idx, tech_idx]
        p_nom_max = profiles_ds['p_nom_max'].values[bus_idx, tech_idx]
        avg_cf = profiles_ds['avg_cf'].values[bus_idx, tech_idx]
        
        # Categorize issues
        if np.isnan(avg_cf) or avg_cf <= 0:
            n_missing_cf += 1
        elif np.isnan(potential) or potential <= 0:
            n_zero_potential += 1
        elif np.isnan(p_nom_max) or p_nom_max <= 0:
            n_missing_p_nom_max += 1
        else:
            n_valid_entries += 1

total_entries = len(profiles_ds.bus) * len(profiles_ds.technology)

logger.info(f"\nTotal entries: {total_entries}")
logger.info(f"  Valid entries:              {n_valid_entries:6d} ({100*n_valid_entries/total_entries:5.1f}%)")
logger.info(f"  Missing/zero CF:            {n_missing_cf:6d} ({100*n_missing_cf/total_entries:5.1f}%)")
logger.info(f"  Zero/missing potential:     {n_zero_potential:6d} ({100*n_zero_potential/total_entries:5.1f}%)")
logger.info(f"  Zero/missing p_nom_max:     {n_missing_p_nom_max:6d} ({100*n_missing_p_nom_max/total_entries:5.1f}%)")

# Verify sums to 100%
accounted = n_valid_entries + n_missing_cf + n_zero_potential + n_missing_p_nom_max
logger.info(f"\nAccounted for: {accounted}/{total_entries}")

logger.info("\n" + "="*70)
logger.info("COVERAGE BY BUS")
logger.info("="*70)

# Count buses with full coverage (all 3 techs valid)
buses_full_coverage = 0
buses_partial_coverage = 0
buses_no_coverage = 0

for bus_idx in range(len(profiles_ds.bus)):
    valid_count = 0
    for tech_idx in range(len(profiles_ds.technology)):
        potential = profiles_ds['potential'].values[bus_idx, tech_idx]
        p_nom_max = profiles_ds['p_nom_max'].values[bus_idx, tech_idx]
        avg_cf = profiles_ds['avg_cf'].values[bus_idx, tech_idx]
        
        if (not (np.isnan(avg_cf) or avg_cf <= 0 or np.isnan(potential) or potential <= 0 
                 or np.isnan(p_nom_max) or p_nom_max <= 0)):
            valid_count += 1
    
    if valid_count == 3:
        buses_full_coverage += 1
    elif valid_count > 0:
        buses_partial_coverage += 1
    else:
        buses_no_coverage += 1

n_buses = len(profiles_ds.bus)
logger.info(f"Buses with full coverage (all 3 techs):    {buses_full_coverage:6d} ({100*buses_full_coverage/n_buses:5.1f}%)")
logger.info(f"Buses with partial coverage (1-2 techs):   {buses_partial_coverage:6d} ({100*buses_partial_coverage/n_buses:5.1f}%)")
logger.info(f"Buses with no coverage (0 techs):          {buses_no_coverage:6d} ({100*buses_no_coverage/n_buses:5.1f}%)")

logger.info("\n" + "="*70)
logger.info("CONCLUSION")
logger.info("="*70)
logger.info(f"✓ Dataset has {100*n_valid_entries/total_entries:.1f}% valid data")
logger.info(f"✓ {100*buses_full_coverage/n_buses:.1f}% of buses have complete technology coverage")
logger.info("✓ Dataset is suitable for downstream analysis")
logger.info("="*70)


2026-04-10 11:27:36,561 - __main__ - INFO - ======================================================================
2026-04-10 11:27:36,562 - __main__ - INFO - DATA QUALITY ANALYSIS - COVERAGE REPORT
2026-04-10 11:27:36,562 - __main__ - INFO - ======================================================================
2026-04-10 11:27:36,984 - __main__ - INFO - 
Total entries: 9441
2026-04-10 11:27:36,985 - __main__ - INFO -   Valid entries:                3222 ( 34.1%)
2026-04-10 11:27:36,985 - __main__ - INFO -   Missing/zero CF:              5177 ( 54.8%)
2026-04-10 11:27:36,986 - __main__ - INFO -   Zero/missing potential:       1042 ( 11.0%)
2026-04-10 11:27:36,987 - __main__ - INFO -   Zero/missing p_nom_max:          0 (  0.0%)
2026-04-10 11:27:36,988 - __main__ - INFO - 
Accounted for: 9441/9441
2026-04-10 11:27:36,989 - __main__ - INFO - 
2026-04-10 11:27:36,989 - __main__ - INFO - COVERAGE BY BUS
2026-04-10 11:27:36,990 - __main__ - INFO - ==========================================

In [ ]:
"""
CLUSTERING FUNCTION (CURRENTLY DISABLED)

The function below implements geographic clustering to aggregate per-bus profiles
into clusters. This is not used in the current production pipeline.
To enable clustering, uncomment the function and execution cells below.
"""

# def cluster_renewable_potentials(
#     tech_profiles_nc,
#     onshore_regions_gpd,
#     offshore_regions_gpd,
#     country_codes=None,
#     total_clusters=10,
#     onshore_ratio=0.8,
#     random_state=42,
#     separate_onshore_offshore=True,
# ):
#     """
#     Cluster renewable resource potentials and capacity factors by geographic proximity.
#     
#     This function aggregates per-region data into fewer geographic clusters using k-means
#     clustering on projected coordinates. NOT USED in current production pipeline.
#     """
#     pass  # Function implementation hidden - see git history for full code

## OPTIONAL: Geographic Clustering (Currently Disabled)

The following cells implement optional geographic clustering to aggregate regions into fewer clusters. This is not used in the current production pipeline, which operates at the per-bus level for maximum granularity. Uncomment these sections if clustering is needed for downstream workflows.


In [ ]:
# ===== CLUSTERING: Execute (DISABLED) =====
# clustering_not_used = True  # Set to True to skip clustering operations
# if not clustering_not_used:
#     logger.info("CLUSTERING WORKFLOW (currently disabled)")
#     clusters_xr, cluster_metadata = cluster_renewable_potentials(
#         tech_profiles_nc,
#         onshore_regions_gpd,
#         offshore_regions_gpd,
#         total_clusters=10,
#         onshore_ratio=0.8,
#         separate_onshore_offshore=True,
#     )

..\..\AppData\Local\Temp\ipykernel_10284\3139860479.py - ✓ Cluster allocation valid: 8 onshore + 2 offshore = 10 total
..\..\AppData\Local\Temp\ipykernel_10284\3139860479.py - Clustering 3 countries: ['IRL', 'NLD', 'PRT']
..\..\AppData\Local\Temp\ipykernel_10284\3139860479.py - Separate onshore/offshore: True
..\..\AppData\Local\Temp\ipykernel_10284\3139860479.py - 
IRL: 296 regions
..\..\AppData\Local\Temp\ipykernel_10284\3139860479.py -   Clustering onshore: 228 regions → 8 clusters
..\..\AppData\Local\Temp\ipykernel_10284\3139860479.py -   Clustering offshore: 68 regions → 2 clusters
..\..\AppData\Local\Temp\ipykernel_10284\3139860479.py - 
NLD: 346 regions
..\..\AppData\Local\Temp\ipykernel_10284\3139860479.py -   Clustering onshore: 291 regions → 8 clusters
..\..\AppData\Local\Temp\ipykernel_10284\3139860479.py -     offwind-ac: No regions in cluster NLD_ON_01
..\..\AppData\Local\Temp\ipykernel_10284\3139860479.py -     offwind-ac: No regions in cluster NLD_ON_02
..\..\AppData\Loc

In [ ]:
# ===== CLUSTERING: Validate (DISABLED) =====
# if not clustering_not_used:
#     def validate_clustering(
#         cluster_metadata,
#         clusters_xr,
#         tech_profiles_nc,
#         onshore_regions_gpd,
#         offshore_regions_gpd,
#     ):
#         """Validates clustering by comparing raw vs clustered potentials."""
#         pass  # Implementation hidden - see git history for full code
#     
#     validation_results = validate_clustering(
#         cluster_metadata,
#         clusters_xr,
#         tech_profiles_nc,
#         onshore_regions_gpd,
#         offshore_regions_gpd,
#     )

In [ ]:
# # ===== CLUSTERING: Validate Results (DISABLED) =====
# # if not clustering_not_used:
# #     validation_results = validate_clustering(
# #         cluster_metadata,
# #         clusters_xr,
# #         tech_profiles_nc,
# #         onshore_regions_gpd,
# #         offshore_regions_gpd,
# #     )


[VALIDATION] 9 entries | 9 perfect | Max diff: 0.0000 GW | ✓ PASS



In [ ]:
# ===== CLUSTERING: Save Results (DISABLED) =====
# if not clustering_not_used:
#     def save_clusters(
#         clusters_xr,
#         cluster_metadata,
#         output_dir="data",
#         filename_prefix="renewable_clusters",
#     ):
#         """Save clustered renewable potential data to NetCDF and metadata to CSV."""
#         pass  # Implementation hidden - see git history for full code
#     
#     save_paths = save_clusters(clusters_xr, cluster_metadata, output_dir="data")
#     logger.info(f"Clustering results saved: {save_paths}")

..\..\AppData\Local\Temp\ipykernel_10284\1196717265.py - ✓ Saved clustered potentials: data\renewable_clusters_20260325_172014.nc
..\..\AppData\Local\Temp\ipykernel_10284\1196717265.py - ✓ Saved cluster metadata: data\renewable_clusters_metadata_20260325_172014.csv
